# Análise do PBIA — Plano Brasileiro de Inteligência Artificial

Este notebook integra uma análise acadêmica de rigor, desenvolvida no âmbito de uma pesquisa em Relações Internacionais dedicada ao estudo comparado das estratégias nacionais de inteligência artificial. Seu objeto específico é o Plano Brasileiro de Inteligência Artificial (PBIA), documento que consolida a estratégia do governo federal brasileiro para o fomento, a regulação e o uso da inteligência artificial no país.

A análise aqui conduzida busca compreender, de modo simultaneamente **quantitativo e qualitativo**, a linguagem empregada pelo documento — os termos, categorias, ênfases retóricas, valores e prioridades estratégicas que estruturam o texto —, de modo a identificar os marcos conceituais e os campos semânticos por meio dos quais o Brasil formula sua política de inteligência artificial.

A partir dessa leitura, pretende-se **posicionar o documento internacionalmente**, comparando a linguagem e as escolhas discursivas de o Brasil com o vocabulário e as ênfases adotados pelos demais países e blocos contemplados neste projeto (Brasil, China, Estados Unidos, Europa e Índia), de modo a mapear convergências, divergências e posicionamentos estratégicos distintivos no debate internacional sobre desenvolvimento e governança de inteligência artificial.

As análises e visualizações produzidas neste notebook seguem as diretrizes metodológicas da **Skill02** (Análise e Visualização Gráfica de Documentos): baseiam-se exclusivamente no campo `texto_completo` do JSON de extração correspondente a este documento, produzido na etapa anterior (Skill01), adotam rigor acadêmico integral na leitura do texto-fonte, evitam generalizações, simplificações e inferências não fundamentadas no texto original, e cada visualização construída é acompanhada de sua respectiva descrição, leitura, interpretação e eventuais limitações metodológicas.


## Análise de Vocabulário — Skill02 + Skill 02_Análise_Vocab_A

Esta seção aplica o **Passo 02** (Skill02 – Análise e Visualização Gráfica de Documentos) em conjunto com o seu complemento especializado, a **Skill 02_Análise_Vocab_A** (Análise de Vocabulário e Termos), a pedido do usuário, para levantar os principais termos e o vocabulário utilizados pelo PBIA.

Do JSON `pbia.json`, são utilizados **exclusivamente** os campos `titulo`, `pais_ou_bloco` e `texto_completo`, conforme o Protocolo de uso do JSON da Skill02. Os campos `elementos_descartados`, `data_extracao`, `data_publicacao` e `fonte` não entram nesta análise — em especial, `elementos_descartados` não tem qualquer valor analítico aqui, servindo apenas de auditoria da Skill01.

A metodologia completa (remoção de stopwords, tratamento de siglas e expressões compostas, agrupamento de variantes morfológicas e critério de corte) está documentada de forma auditável e cumulativa no registro persistente [`pbia_vocab_registro.md`](./pbia_vocab_registro.md), nesta mesma pasta, conforme exigido pelo item 6 da Skill 02_Análise_Vocab_A. Esse registro deve ser consultado — e atualizado — antes de qualquer nova visualização de vocabulário sobre este mesmo documento.

In [ ]:
import json
import re
from collections import Counter

import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

with open("pbia.json", encoding="utf-8") as f:
    pbia = json.load(f)

# Protocolo de uso do JSON (Skill02): apenas estes três campos são utilizados nesta análise
titulo = pbia["titulo"]
pais_ou_bloco = pbia["pais_ou_bloco"]
texto_completo = pbia["texto_completo"]

print(f"Documento: {titulo}")
print(f"País/bloco: {pais_ou_bloco}")
print(f"Tamanho de texto_completo: {len(texto_completo):,} caracteres".replace(",", "."))

### 1. Identificação do idioma e etapas metodológicas (Skill 02_Análise_Vocab_A)

`texto_completo` está integralmente em **inglês** (a publicação oficial do MCTI/CGEE aqui analisada foi extraída em sua versão em língua inglesa — ver Skill01). As stopwords e regras de normalização abaixo são, portanto, do inglês.

> **Revisão metodológica (2026-08-05).** Por determinação do usuário, este documento — que já preservava hífens como parte do token desde a criação (diferente dos demais documentos do projeto, que tratavam hífen como separador universal) — recebeu: (i) auditoria formal das 58 formas hifenizadas do texto (todas mantidas unidas); (ii) três novos bigramas de desambiguação contextual, corrigindo polissemias genuínas em "power" (computacional vs. jurídico-institucional) e "state"/"states" (subnacional genérico vs. "the State" como ente político abstrato vs. "United States"), além de "generation" (título de eixo do Plano vs. geração de conteúdo por IA vs. sentido genérico); (iii) a primeira normalização de famílias verbais deste documento (o registro original só normalizava substantivos); e (iv) expansão do corte de Top 25 para **Top 50**. O registro persistente documenta a versão anterior e a mudança em detalhe.

O bloco de código a seguir implementa, nesta ordem:
1. **Expressões compostas, siglas equivalentes e desambiguações contextuais** — tratadas como unidade única *antes* da tokenização em palavras isoladas, para não fragmentar termos técnicos nem misturar referentes distintos sob a mesma grafia (ex.: "Artificial Intelligence"/"AI" nunca é contado como as palavras soltas "artificial" + "intelligence"; "United States" nunca se mistura com "state(s)" subnacional).
2. **Tokenização** do texto em palavras, preservando hífens internos como parte do token (Skill 02_Análise_Vocab_A, item 4) — cada composto hifenizado é auditado individualmente no registro persistente.
3. **Remoção** de stopwords do inglês, de marcadores de lista/resíduos não semânticos e do termo de baixo valor analítico "expected" (ver justificativas no registro persistente).
4. **Normalização morfológica** (singular/plural do mesmo lema, e — nesta revisão, pela primeira vez — famílias de conjugações verbais), agrupando variantes sob um rótulo representativo comum, sem descartar nenhuma ocorrência do cômputo.


In [ ]:
# ============ 1) EXPRESSÕES COMPOSTAS, SIGLAS E DESAMBIGUAÇÕES CONTEXTUAIS ============
# Ordem: da mais específica/longa para a mais genérica/curta, para evitar
# que um padrão curto "consuma" parte de um padrão mais longo antes da hora.
COMPOUND_TERMS = [
    (r"\bResearch and Development\b|\bR&D\b", "ZZCMPRESEARCHDEV", "Research & Development (R&D)"),
    (r"\bUnified Health System\b|\bSUS\b", "ZZCMPSUS", "Unified Health System (SUS)"),
    (r"\bNational Data Infrastructure\b|\bIND\b", "ZZCMPIND", "National Data Infrastructure (IND)"),
    (r"\bLarge Language Models?\b|\bLLMs?\b", "ZZCMPLLM", "Large Language Models (LLM)"),
    (r"\bSustainable Development Goals?\b|\bSDGs?\b", "ZZCMPSDG", "Sustainable Development Goals (SDGs)"),
    (r"\bArtificial Intelligence\b|\bAI\b", "ZZCMPAI", "Artificial Intelligence (AI)"),
    (r"\bData Centers?\b", "ZZCMPDATACENTER", "Data Center(s)"),
    (r"\bData Infrastructure\b", "ZZCMPDATAINFRA", "Data Infrastructure"),
    (r"\bPublic Sector\b", "ZZCMPPUBSECTOR", "Public Sector"),
    (r"\bPublic Services?\b", "ZZCMPPUBSERVICE", "Public Service(s)"),
    (r"\bPrivate Sector\b", "ZZCMPPRIVSECTOR", "Private Sector"),
    (r"\bValue Chains?\b", "ZZCMPVALUECHAIN", "Value Chain"),
    (r"\bMachine Learning\b", "ZZCMPMACHLEARN", "Machine Learning"),
    (r"\bFederal Government\b", "ZZCMPFEDGOV", "Federal Government"),
    (r"\bDigital Government\b", "ZZCMPDIGGOV", "Digital Government"),
    (r"\bPublic Administration\b", "ZZCMPPUBADMIN", "Public Administration"),
    (r"\bClean Energy\b", "ZZCMPCLEANENERGY", "Clean Energy"),
    (r"\bEnergy Matrix\b", "ZZCMPENERGYMATRIX", "Energy Matrix"),
]

# Desambiguação contextual (registro persistente, Seções 1.5/2.5): bigramas/trechos que
# resolvem polissemias genuínas de "power", "state(s)" e "generation" identificadas por
# concordância. Os três primeiros usam maiúscula distintiva (case-sensitive, sem re.I).
CONTEXT_DISAMBIGUATION = [
    (r"\bUnited States\b", "ZZCMPUNITEDSTATES", "United States", False),
    (r"\bthe State\b", "ZZCMPSTATEENTITY", "the State (ente político/entidade abstrata)", False),
    (r"\bFederal Public Power\b", "ZZCMPPUBLICPOWER", "Poder Público Federal (termo jurídico-institucional)", False),
    (r"\bGeneration of national\b", "ZZCMPGENAXIS", "Generation of national capacities/capabilities (título de eixo do Plano)", False),
    (r"\bcomputational power\b", "ZZCMPPOWERCOMPUTE", "power (computacional)", True),
    (r"\bcontent generation\b", "ZZCMPGENCONTENT", "generation (geração de conteúdo por IA)", True),
]

work_text = texto_completo
placeholder_map = {}
for pattern, placeholder, label in COMPOUND_TERMS:
    placeholder_map[placeholder.lower()] = label
    work_text = re.sub(pattern, f" {placeholder} ", work_text, flags=re.IGNORECASE)
for pattern, placeholder, label, case_insensitive in CONTEXT_DISAMBIGUATION:
    placeholder_map[placeholder.lower()] = label
    flags = re.IGNORECASE if case_insensitive else 0
    work_text = re.sub(pattern, f" {placeholder} ", work_text, flags=flags)

# ============ 2) TOKENIZAÇÃO ============
# Hífen preservado como parte do token desde a criação deste notebook (não é a regra
# "hífen = separador" corrigida nos demais documentos do projeto). Auditoria formal das
# 58 formas hifenizadas identificadas: todas mantidas unidas (registro persistente, 1.4).
raw_tokens = re.findall(r"[A-Za-zÀ-ÖØ-öø-ÿ][A-Za-zÀ-ÖØ-öø-ÿ'\-]*", work_text)
tokens = [t.lower() for t in raw_tokens]

# ============ 3) REMOÇÃO (exclusão) ============
ARTICLES = {"a", "an", "the"}
PREPOSITIONS = {"of","in","to","for","with","on","by","as","at","from","into","about","through",
    "during","before","after","above","below","between","under","over","without","within","among",
    "throughout","towards","toward","upon","across","per","via","despite","unlike","regarding","off",
    "out","up","down","besides"}
CONJUNCTIONS = {"and","or","but","nor","so","yet","if","because","while","although","though",
    "whether","since","unless","until","than"}
PRONOUNS_DETERMINERS = {"it","its","this","that","these","those","we","our","ours","they","their",
    "theirs","which","who","whom","whose","i","you","he","she","him","her","us","them","his","hers",
    "itself","themselves","ourselves","yourself","yourselves","himself","herself","one","ones",
    "such","other","others","any","some","each","all","both","either","neither","no","none","own",
    "same"}
AUX_MODAL_VERBS = {"is","are","was","were","be","been","being","am","has","have","had","do","does",
    "did","will","would","shall","should","can","could","may","might","must","ought"}
GENERIC_CONNECTORS = {"more","most","much","many","few","several","various","not","also","only",
    "just","still","even","well","thus","therefore","however","moreover","furthermore","given",
    "whereas"}
STOPWORDS = (ARTICLES | PREPOSITIONS | CONJUNCTIONS | PRONOUNS_DETERMINERS
             | AUX_MODAL_VERBS | GENERIC_CONNECTORS)

# marcadores de lista/seção (A)/B)/C)/D), I)/II)/III)/IV)/V)) — "a" e "i" já cobertos acima
LIST_MARKERS = {"b", "c", "d", "ii", "iii", "iv", "v"}

# termo de template de baixo valor analítico (ver justificativa no registro persistente)
TEMPLATE_TERMS = {"expected"}

# ============ 4) NORMALIZAÇÃO (agrupamento de variantes, sem exclusão) ============
PLURAL_MERGE = {
    "actions": "action(s)", "action": "action(s)",
    "challenges": "challenge(s)", "challenge": "challenge(s)",
    "impacts": "impact(s)", "impact": "impact(s)",
    "solutions": "solution(s)", "solution": "solution(s)",
    "resources": "resource(s)", "resource": "resource(s)",
    "investments": "investment(s)", "investment": "investment(s)",
    "models": "model(s)", "model": "model(s)",
    "benefits": "benefit(s)", "benefit": "benefit(s)",
    "professionals": "professional(s)", "professional": "professional(s)",
    "networks": "network(s)", "network": "network(s)",
    "initiatives": "initiative(s)", "initiative": "initiative(s)",
    "institutions": "institution(s)", "institution": "institution(s)",
    "researchers": "researcher(s)", "researcher": "researcher(s)",
    "capacities": "capacity/capacities", "capacity": "capacity/capacities",
    "companies": "company/companies", "company": "company/companies",
    "citizens": "citizen(s)", "citizen": "citizen(s)",
    "agencies": "agency/agencies", "agency": "agency/agencies",
    "axes": "axis/axes", "axis": "axis/axes",
    "centers": "center(s)", "center": "center(s)",
    "innovations": "innovation(s)", "innovation": "innovation(s)",
    "qualifications": "qualification(s)", "qualification": "qualification(s)",
    "systems": "system(s)", "system": "system(s)",
    "technologies": "technology/technologies", "technology": "technology/technologies",
    "sectors": "sector(s)", "sector": "sector(s)",
    "processes": "process(es)", "process": "process(es)",
    "services": "service(s)", "service": "service(s)",
    "governments": "government(s)", "government": "government(s)",
    "policies": "policy/policies", "policy": "policy/policies",
    "rights": "right(s)", "right": "right(s)",
    "risks": "risk(s)", "risk": "risk(s)",
    "servants": "servant(s)", "servant": "servant(s)",
    "brazil's": "brazil",
    # -- novos pares (cobertura ampliada para Top 50, revisão 2026-08-05) --
    "applications": "application(s)", "application": "application(s)",
    "areas": "area(s)", "area": "area(s)",
    "advantages": "advantage(s)", "advantage": "advantage(s)",
    "frameworks": "framework(s)", "framework": "framework(s)",
    "databases": "database(s)", "database": "database(s)",
    "decisions": "decision(s)", "decision": "decision(s)",
    "ecosystems": "ecosystem(s)", "ecosystem": "ecosystem(s)",
    "enterprises": "enterprise(s)", "enterprise": "enterprise(s)",
    "environments": "environment(s)", "environment": "environment(s)",
    "examples": "example(s)", "example": "example(s)",
    "funds": "fund(s)", "fund": "fund(s)",
    "goals": "goal(s)", "goal": "goal(s)",
    "humans": "human(s)", "human": "human(s)",
    "interests": "interest(s)", "interest": "interest(s)",
    "jobs": "job(s)", "job": "job(s)",
    "leaders": "leader(s)", "leader": "leader(s)",
    "levels": "level(s)", "level": "level(s)",
    "markets": "market(s)", "market": "market(s)",
    "missions": "mission(s)", "mission": "mission(s)",
    "nations": "nation(s)", "nation": "nation(s)",
    "objectives": "objective(s)", "objective": "objective(s)",
    "partnerships": "partnership(s)", "partnership": "partnership(s)",
    "plans": "plan(s)", "plan": "plan(s)",
    "platforms": "platform(s)", "platform": "platform(s)",
    "populations": "population(s)", "population": "population(s)",
    "problems": "problem(s)", "problem": "problem(s)",
    "projects": "project(s)", "project": "project(s)",
    "scenarios": "scenario(s)", "scenario": "scenario(s)",
    "schools": "school(s)", "school": "school(s)",
    "sciences": "science(s)", "science": "science(s)",
    "standards": "standard(s)", "standard": "standard(s)",
    "students": "student(s)", "student": "student(s)",
    "supercomputers": "supercomputer(s)", "supercomputer": "supercomputer(s)",
    "tools": "tool(s)", "tool": "tool(s)",
    "transformations": "transformation(s)", "transformation": "transformation(s)",
    "treatments": "treatment(s)", "treatment": "treatment(s)",
    "volumes": "volume(s)", "volume": "volume(s)",
    "contexts": "context(s)", "context": "context(s)",
    "characteristics": "characteristic(s)", "characteristic": "characteristic(s)",
    "biases": "bias(es)", "bias": "bias(es)",
    "businesses": "business(es)", "business": "business(es)",
    "costs": "cost(s)", "cost": "cost(s)",
    "changes": "change(s)", "change": "change(s)",
    "results": "result(s)", "result": "result(s)",
    # "state(s)" genérico/subnacional — após extração de "United States" e "the State"
    # (Seção 1.5 do registro), o residual designa estados federativos/uso genérico
    "states": "state(s) (subnacional/genérico)", "state": "state(s) (subnacional/genérico)",
}
# "country" (singular, = Brasil no estilo do texto) e "countries" (plural, = outras nações)
# foram deliberadamente NÃO agrupados: designam referentes distintos (ver registro persistente).

# Novidade desta revisão: famílias de conjugações verbais (o registro original só
# normalizava substantivos). Mantidas separadas dos substantivos derivacionais
# correspondentes quando existentes (ex.: develop/development; create/creation).
VERB_MERGE = {
    "promote": "promote", "promotes": "promote", "promoting": "promote", "promoted": "promote",
    "increase": "increase", "increases": "increase", "increasing": "increase", "increased": "increase",
    "improve": "improve", "improves": "improve", "improving": "improve",
    "support": "support (verbo)", "supports": "support (verbo)", "supporting": "support (verbo)", "supported": "support (verbo)",
    "ensure": "ensure", "ensures": "ensure", "ensuring": "ensure", "ensured": "ensure",
    "strengthen": "strengthen", "strengthens": "strengthen", "strengthening": "strengthen", "strengthened": "strengthen",
    "foster": "foster", "fosters": "foster", "fostering": "foster", "fostered": "foster",
    "establish": "establish", "establishes": "establish", "establishing": "establish", "established": "establish",
    "expand": "expand", "expands": "expand", "expanding": "expand", "expanded": "expand",
    "create": "create (verbo)", "creates": "create (verbo)", "creating": "create (verbo)", "created": "create (verbo)",
    "develop": "develop (verbo)", "develops": "develop (verbo)", "developing": "develop (verbo)", "developed": "develop (verbo)",
    "implement": "implement (verbo)", "implements": "implement (verbo)", "implementing": "implement (verbo)", "implemented": "implement (verbo)",
    "include": "include", "includes": "include", "including": "include", "included": "include",
    "identify": "identify", "identifies": "identify", "identifying": "identify", "identified": "identify",
    "require": "require", "requires": "require", "requiring": "require", "required": "require",
    "guarantee": "guarantee", "guarantees": "guarantee", "guaranteeing": "guarantee",
    "monitor": "monitor", "monitors": "monitor", "monitoring": "monitor", "monitored": "monitor",
    "integrate": "integrate", "integrates": "integrate", "integrating": "integrate", "integrated": "integrate",
    "propose": "propose", "proposes": "propose", "proposing": "propose", "proposed": "propose",
    "protect": "protect", "protects": "protect", "protecting": "protect", "protected": "protect",
    "provide": "provide (verbo)", "provides": "provide (verbo)", "providing": "provide (verbo)", "provided": "provide (verbo)",
    "reduce": "reduce", "reduces": "reduce", "reducing": "reduce", "reduced": "reduce",
    "launch": "launch", "launches": "launch", "launching": "launch", "launched": "launch",
    "offer": "offer", "offers": "offer", "offering": "offer", "offered": "offer",
}

canon = {**PLURAL_MERGE, **VERB_MERGE}

freq = Counter()
for t in tokens:
    if t in placeholder_map:
        freq[placeholder_map[t]] += 1
        continue
    if len(t) == 1:
        continue
    if t in LIST_MARKERS:
        continue
    if t in STOPWORDS:
        continue
    if t in TEMPLATE_TERMS:
        continue
    freq[canon.get(t, t)] += 1

print(f"Tokens de conteúdo após o pipeline completo: {sum(freq.values()):,}".replace(",", "."))
print(f"Termos únicos após o pipeline completo: {len(freq):,}".replace(",", "."))


In [ ]:
TOP_N = 50  # critério de corte padrão (Skill 02_Análise_Vocab_A, item 6, revisão 2026-08-05)

top50 = freq.most_common(TOP_N)
top25 = top50[:25]
df_top50 = pd.DataFrame(top50, columns=["termo", "frequência"])
df_top50.index = range(1, len(df_top50) + 1)
df_top50.index.name = "ranking"
df_top50


In [ ]:
terms = [t for t, n in top25][::-1]
counts = [n for t, n in top25][::-1]

# Paleta validada (skill "dataviz" deste ambiente): slot 1 azul / slot 2 laranja,
# par categórico com CVD ΔE 9.1 (claro) / 8.4 (escuro) — acima do alvo de 8.
COLOR_MAIN = "#2a78d6"
COLOR_HIGHLIGHT = "#eb6834"
colors = [COLOR_HIGHLIGHT if t == "Artificial Intelligence (AI)" else COLOR_MAIN for t in terms]

fig, ax = plt.subplots(figsize=(10, 11.5), dpi=150)
bars = ax.barh(terms, counts, color=colors, height=0.68, zorder=3)

for rect, val in zip(bars, counts):
    ax.text(rect.get_width() + max(counts) * 0.01, rect.get_y() + rect.get_height() / 2,
             f"{val}", va="center", ha="left", fontsize=9, color="#33322f")

ax.set_xlabel("Frequência (nº de ocorrências em texto_completo)", fontsize=10, color="#33322f")

fig.suptitle("PBIA – Termos mais frequentes do vocabulário (Top 25)",
             fontsize=13.5, fontweight="bold", color="#0b0b0b", x=0.02, y=0.99, ha="left")
fig.text(0.02, 0.965,
         "Após remoção de stopwords, normalização morfológica/verbal e de siglas/expressões compostas/desambiguações "
         "— Skill02 + Skill 02_Análise_Vocab_A",
         fontsize=8.5, color="#6b6a66", ha="left")

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_visible(False)
ax.spines["bottom"].set_color("#c9c8c2")
ax.tick_params(axis="y", length=0, labelsize=9.5)
ax.tick_params(axis="x", labelsize=9)
ax.xaxis.grid(True, color="#e6e5e0", zorder=0)
ax.set_axisbelow(True)
ax.set_xlim(0, max(counts) * 1.12)

legend_elems = [
    Patch(facecolor=COLOR_HIGHLIGHT, label="Artificial Intelligence (AI) — termo dominante e esperado"),
    Patch(facecolor=COLOR_MAIN, label="Demais termos (posições 2–25)"),
]
ax.legend(handles=legend_elems, loc="lower right", frameon=False, fontsize=8.5)

plt.tight_layout(rect=[0, 0, 1, 0.955])
plt.show()


### Visão expandida — Top 50

Por determinação do usuário, a análise passa a cobrir também o **Top 50** (o Top 25 acima é mantido por continuidade). A expansão é sustentada pela cobertura ampliada de normalização morfológica — 68 pares nominais (29 já existentes + 39 novos) — e, pela primeira vez neste documento, 22 famílias de conjugações verbais, descritas no registro persistente, Seção 2.3.


In [ ]:
terms50 = [t for t, n in top50][::-1]
counts50 = [n for t, n in top50][::-1]
colors50 = [COLOR_HIGHLIGHT if t == "Artificial Intelligence (AI)" else (COLOR_MAIN if i >= len(top50) - 25 else "#a9c8ec")
            for i, t in enumerate(terms50)]

fig, ax = plt.subplots(figsize=(10.5, 19), dpi=150)
bars = ax.barh(terms50, counts50, color=colors50, height=0.72, zorder=3)

for rect, val in zip(bars, counts50):
    ax.text(rect.get_width() + max(counts50) * 0.01, rect.get_y() + rect.get_height() / 2,
             f"{val}", va="center", ha="left", fontsize=8.5, color="#33322f")

ax.set_xlabel("Frequência (nº de ocorrências em texto_completo)", fontsize=10, color="#33322f")

fig.suptitle("PBIA – Termos mais frequentes do vocabulário (Top 50, visão expandida)",
             fontsize=13.5, fontweight="bold", color="#0b0b0b", x=0.02, y=0.99, ha="left")
fig.text(0.02, 0.975,
         "Após remoção de stopwords, normalização morfológica/verbal e de siglas/expressões compostas/desambiguações "
         "— Skill02 + Skill 02_Análise_Vocab_A",
         fontsize=8.5, color="#6b6a66", ha="left")

ax.axhline(24.5, color="#999999", linestyle="--", linewidth=0.9, zorder=4)
ax.text(max(counts50) * 0.98, 25.3, "Top 25", ha="right", fontsize=8.5, color="#666666", style="italic")

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_visible(False)
ax.spines["bottom"].set_color("#c9c8c2")
ax.tick_params(axis="y", length=0, labelsize=9)
ax.tick_params(axis="x", labelsize=9)
ax.xaxis.grid(True, color="#e6e5e0", zorder=0)
ax.set_axisbelow(True)
ax.set_xlim(0, max(counts50) * 1.12)

legend_elems = [
    Patch(facecolor=COLOR_HIGHLIGHT, label="Artificial Intelligence (AI) — termo dominante e esperado"),
    Patch(facecolor=COLOR_MAIN, label="Posições 2–25"),
    Patch(facecolor="#a9c8ec", label="Posições 26–50 (visão expandida)"),
]
ax.legend(handles=legend_elems, loc="lower right", frameon=False, fontsize=8.5)

plt.tight_layout(rect=[0, 0, 1, 0.965])
plt.show()


### 2. Análise final (Skill02, item 4 · Skill 02_Análise_Vocab_A, item 10)

**O que foi construído.** Dois gráficos de barras horizontais: Top 25 (mantido por continuidade, com "Artificial Intelligence (AI)" isolado em laranja) e Top 50 (visão expandida, posições 26–50 em azul claro), após a aplicação integral do pipeline metodológico descrito acima e detalhado, de forma auditável, em [`pbia_vocab_registro.md`](./pbia_vocab_registro.md).

**Como ler os gráficos.** Cada barra corresponde a um termo (ou a um grupo de variantes equivalentes) e seu comprimento é o número de ocorrências desse termo em todo o corpo do texto extraído.

**O que mudou com a correção metodológica (comparação com a versão anterior).** Diferentemente dos demais documentos do projeto, o PBIA já preservava hífens desde a criação — não houve, portanto, mudança nos números por conta da tokenização de hífen. As mudanças reais vieram de três fontes: (i) a desambiguação de "power" (6 ocorrências computacionais isoladas de 1 ocorrência jurídico-institucional, "Federal Public Power"); (ii) a desambiguação de "state(s)" (6 ocorrências subnacionais isoladas de 2 de "the State" como ente político e 3 de "United States"); e (iii) a introdução, pela primeira vez, de famílias verbais normalizadas — o que faz aparecer no Top 25/50 termos como "increase" (74), "support (verbo)" (51), "promote" (50) e "develop (verbo)" (46), antes dispersos em suas formas flexionadas separadas (ex.: "increase"/"increases"/"increasing"/"increased" contados individualmente).

**Interpretação à luz do conteúdo do documento.** "Artificial Intelligence (AI)" domina com folga (627 ocorrências) — resultado esperado e tautológico para um plano nacional inteiramente dedicado a IA. Entre as posições 2–25, o vocabulário do PBIA revela quatro camadas: (i) **termos de ação/estrutura de política** — "development", "action(s)", "impact(s)", "challenge(s)", "solution(s)" — parcialmente inflados pelos rótulos fixos dos Anexos 1-2 (ver registro persistente); (ii) **verbos de fomento**, agora visíveis como categoria própria — "increase" (74), "support (verbo)" (51), "promote" (50), "develop (verbo)" (46) — que caracterizam o PBIA como documento de indução e capacitação, não de comando regulatório direto; (iii) **termos de soberania/identidade nacional** — "brazil" (121), "national" (100), "brazilian" (95) — que somam 316 ocorrências, mais do que qualquer termo isolado depois de "AI"; e (iv) **termos técnico-setoriais** — "technology/technologies", "technological", "data", "innovation(s)", "system(s)", "infrastructure", "research", "education", "health".

**Achados da visão expandida (Top 50).** "reduction" (37), "governance" (36), "management" (35), "efficiency" (31) e "responsible" (31), visíveis apenas no Top 50, reforçam um eixo de governança responsável e eficiência administrativa que o corte de 25 não deixava aparente; "country" (33), mantido deliberadamente separado de "countries" (ver registro), confirma o uso sistemático de "the Country" como autorreferência ao próprio Brasil.

**Ajustes possíveis.** (a) Segunda versão em escala logarítmica ou com exclusão pontual de "AI", para melhor discriminar as diferenças entre os termos de posições 2 em diante; (b) expandir a lista de expressões compostas tratadas como unidade (ex.: "deep learning", "sustainable development", "public health") caso análises futuras demandem maior granularidade.

**Limitações e fraquezas metodológicas.**
- O critério de fusão de expressões compostas depende de adjacência textual exata (bigrama); variantes não adjacentes do mesmo conceito não são capturadas.
- A cobertura da normalização morfológica e verbal, e da desambiguação contextual, foi concentrada nos candidatos plausíveis ao corte de Top 50 — não há lematização/desambiguação exaustiva de toda a cauda longa de 2.280 termos distintos do vocabulário final.
- A contagem é de frequência absoluta, não relativa/ponderada por seção — adequado para análise individual (sem comparação entre corpora de tamanhos distintos, o que exigiria a Skill 02_Análise_Vocab_B), mas uma limitação a ter em mente na leitura.
- Parte de "action(s)"/"impact(s)"/"challenge(s)" segue inflada pelos rótulos de campo dos Anexos 1-2 (quantificação exata no registro persistente), decisão mantida por esses termos designarem categorias conceituais reais do Plano, não apenas rótulos de template.